# 프로젝트 3 - Weekend 3: Adaptive / Self / Corrective RAG 통합 — solution

**프로젝트**: 법률 문서 기반 검색 에이전트 시스템 (최종 Weekend 3/3)

**목표**:
1. 질문 난이도별로 검색 전략을 분기 (easy→quick, hard→thorough)
2. 검색 문서의 관련성을 평가하고, 답변 품질 기준 미달 시 재생성
3. 로컬 검색 결과를 등급화하고, 부족하면 외부 검색으로 보완
4. 3종 RAG를 하나의 LangGraph로 통합하여 `LegalRAGAgent` 완성

**사전 조건**: Weekend 1, 2 완료 (`LegalGraphAgent`와 벡터스토어가 구축된 상태)

**구성**: 문제 10개 × 약 45분 = 약 8시간

In [ ]:
# 환경 설정
!pip install -q langchain langchain-openai langchain-community faiss-cpu \
    langgraph wikipedia pandas numpy python-dotenv

In [ ]:
import os
import json
import time
from typing import TypedDict, Annotated, Literal, Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage, BaseMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("✅ 환경 설정 완료")

---
## 문제 1: Weekend 2 자산 복원

벡터스토어, 검색 도구, `LegalAgentState`를 재구성하세요. Weekend 3에서는 상태를 **확장**합니다.

**요구사항:**
- `SAMPLE_LAW_ARTICLES`, `vector_store`, `@tool search_law` 재구성
- `RAGAgentState`라는 확장된 TypedDict 정의:
  - `messages`: `Annotated[list[BaseMessage], add_messages]`
  - `query`: `str`
  - `difficulty`: `Literal["easy", "medium", "hard"]` (Adaptive용)
  - `search_results`: `list[dict]`
  - `relevance_scores`: `list[float]` (Self RAG용, 각 문서의 0~1 점수)
  - `grade`: `Literal["pass", "fail", ""]` (Corrective용)
  - `refinement_count`: `int` (재검색 횟수)
  - `answer`: `str`

**평가기준:**
- `RAGAgentState.__annotations__`에 8개 필드 존재
- `search_law` 도구 동작 확인

In [ ]:
# 문제 1: Weekend 2 자산 복원 + RAGAgentState 정의
SAMPLE_LAW_ARTICLES = [
    {"law": "민법", "article": "제750조", "title": "불법행위",
     "content": "고의 또는 과실로 인한 위법행위로 타인에게 손해를 가한 자는 그 손해를 배상할 책임이 있다."},
    {"law": "민법", "article": "제840조", "title": "재판상 이혼원인",
     "content": "부부의 일방은 1.부정한 행위 2.악의의 유기 3.부당한 대우 등의 사유가 있는 경우 이혼을 청구할 수 있다."},
    {"law": "형법", "article": "제307조", "title": "명예훼손",
     "content": "공연히 사실을 적시하여 사람의 명예를 훼손한 자는 2년 이하의 징역이나 500만원 이하의 벌금에 처한다."},
    {"law": "형법", "article": "제329조", "title": "절도",
     "content": "타인의 재물을 절취한 자는 6년 이하의 징역 또는 1천만원 이하의 벌금에 처한다."},
    {"law": "형법", "article": "제347조", "title": "사기",
     "content": "사람을 기망하여 재물의 교부를 받거나 재산상의 이익을 취득한 자는 10년 이하의 징역 또는 2천만원 이하의 벌금에 처한다."},
    {"law": "상법", "article": "제382조", "title": "이사의 선임",
     "content": "이사는 주주총회에서 선임하며, 임기는 3년을 초과하지 못한다."},
]

law_docs = [
    Document(
        page_content=f"{a['law']} {a['article']} ({a['title']}): {a['content']}",
        metadata={"law": a["law"], "article": a["article"], "title": a["title"]},
    )
    for a in SAMPLE_LAW_ARTICLES
]
vector_store = FAISS.from_documents(law_docs, embeddings)

@tool
def search_law(query: str, top_k: int = 3) -> str:
    """법률 조문을 의미 기반으로 검색합니다."""
    results = vector_store.similarity_search(query, k=top_k)
    out = [{"law": r.metadata["law"], "article": r.metadata["article"],
            "title": r.metadata["title"], "content": r.page_content}
           for r in results]
    return json.dumps(out, ensure_ascii=False, indent=2)

# ---- 여기에 코드 작성: RAGAgentState 정의 ----
class RAGAgentState(TypedDict):
    pass

# 검증
print("RAGAgentState 필드:", list(RAGAgentState.__annotations__.keys()) if hasattr(RAGAgentState, '__annotations__') else "❌ 미구현")
print("검색 테스트:", search_law.invoke({"query": "이혼", "top_k": 2})[:150])

In [ ]:
# ✅ 문제 1 정답
class RAGAgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    query: str
    difficulty: Literal["easy", "medium", "hard"]
    search_results: list
    relevance_scores: list
    grade: Literal["pass", "fail", ""]
    refinement_count: int
    answer: str

print("✅ 필드:", list(RAGAgentState.__annotations__.keys()))
print("검색:", search_law.invoke({"query": "이혼", "top_k": 2})[:150])

---
## 문제 2: 질문 난이도 분류기

질문의 복잡도를 `easy / medium / hard` 중 하나로 분류하는 노드를 구현하세요.

**기준:**
- `easy`: 단일 조문 조회로 답변 가능 (예: "명예훼손 처벌은?")
- `medium`: 여러 조문 비교 또는 개념 설명 필요 (예: "절도와 사기 차이?")
- `hard`: 복합 판단·다단계 추론 필요 (예: "배우자 부정행위 후 유기 시 이혼·손해배상 가능한지?")

**요구사항:**
- 함수 시그니처: `difficulty_node(state: RAGAgentState) -> dict`
- LLM 프롬프트에 기준과 예시 포함
- 반환: `{"difficulty": "easy" | "medium" | "hard"}`
- 잘못된 응답 시 `"medium"`으로 fallback

**평가기준:**
- 단순 질문 → easy, 복합 질문 → hard

In [ ]:
# 문제 2: difficulty_node
def difficulty_node(state):
    """질문 난이도를 분류합니다."""
    query = state["query"]
    # ---- 여기에 코드 작성 ----
    # 1) 프롬프트에 easy/medium/hard 기준 + 예시 포함
    # 2) LLM 호출 후 응답 파싱
    # 3) {easy, medium, hard} 아니면 "medium" fallback
    return {"difficulty": "medium"}

# 테스트
test_queries = [
    "명예훼손 처벌은?",  # easy
    "절도와 사기의 차이는?",  # medium
    "배우자가 부정행위 후 유기했을 때 이혼과 손해배상을 함께 청구할 수 있어?",  # hard
]
for q in test_queries:
    state = {"query": q, "difficulty": "", "messages": [], "search_results": [],
             "relevance_scores": [], "grade": "", "refinement_count": 0, "answer": ""}
    print(f"  '{q[:40]}...' → {difficulty_node(state)['difficulty']}")

In [ ]:
# ✅ 문제 2 정답
class _Difficulty(BaseModel):
    level: Literal["easy", "medium", "hard"] = Field(description="질문 난이도")

def difficulty_node(state):
    query = state["query"]
    prompt = f"""다음 질문의 난이도를 분류하세요.

기준:
- easy: 단일 조문 조회로 답변 가능. 예: "명예훼손 처벌은?"
- medium: 여러 조문 비교 또는 개념 설명 필요. 예: "절도와 사기 차이?"
- hard: 복합 판단이나 다단계 추론 필요. 예: "부정행위 후 유기 시 이혼·손해배상 동시 청구 가능?"

질문: {query}"""
    r = llm.with_structured_output(_Difficulty).invoke([HumanMessage(content=prompt)])
    return {"difficulty": r.level}

for q in ["명예훼손 처벌은?", "절도와 사기의 차이는?",
          "배우자가 부정행위 후 유기했을 때 이혼과 손해배상을 함께 청구할 수 있어?"]:
    state = {"query": q, "difficulty": "", "messages": [], "search_results": [],
             "relevance_scores": [], "grade": "", "refinement_count": 0, "answer": ""}
    print(f"  '{q[:40]}...' → {difficulty_node(state)['difficulty']}")

---
## 문제 3: Adaptive Routing — 난이도별 검색 전략

난이도에 따라 검색 k값과 전략을 다르게 적용하는 `adaptive_search_node`를 구현하세요.

**요구사항:**
- 함수 시그니처: `adaptive_search_node(state) -> dict`
- 전략:
  - `easy`: k=2, single search (빠른 답변)
  - `medium`: k=5, single search (여유있게)
  - `hard`: k=5, multi-query — 원 질문 + LLM이 생성한 2개 변형 쿼리로 각각 검색 후 중복 제거
- 반환: `{"search_results": [...]}`

**평가기준:**
- easy: 2개 이하 결과
- hard: 5개 이상 결과 (중복 제거 후)

In [ ]:
# 문제 3: adaptive_search_node
def adaptive_search_node(state):
    """난이도별로 검색 전략을 다르게 적용."""
    query = state["query"]
    difficulty = state["difficulty"]
    # ---- 여기에 코드 작성 ----
    # 1) difficulty에 따라 분기
    # 2) hard인 경우 LLM에게 쿼리 변형 요청 후 멀티 검색 + 중복 제거
    # 3) 결과를 dict 리스트로 반환
    return {"search_results": []}

# 테스트
for q, d in [("명예훼손 처벌은?", "easy"), ("이혼과 손해배상 청구가 함께 가능해?", "hard")]:
    state = {"query": q, "difficulty": d, "messages": [], "search_results": [],
             "relevance_scores": [], "grade": "", "refinement_count": 0, "answer": ""}
    out = adaptive_search_node(state)
    print(f"  [{d}] '{q}' → {len(out['search_results'])}개 문서")

In [ ]:
# ✅ 문제 3 정답
def adaptive_search_node(state):
    query = state["query"]
    difficulty = state["difficulty"]

    if difficulty == "easy":
        k = 2
        queries = [query]
    elif difficulty == "medium":
        k = 5
        queries = [query]
    else:  # hard
        k = 5
        variant_prompt = f"""다음 질문과 의미가 같지만 표현이 다른 검색 쿼리 2개를 생성하세요. 각 줄에 하나씩.

질문: {query}
변형 쿼리:"""
        resp = llm.invoke([HumanMessage(content=variant_prompt)])
        variants = [line.strip(" -•*0123456789.") for line in resp.content.strip().split("\n") if line.strip()][:2]
        queries = [query] + variants

    seen = set()
    results = []
    for q in queries:
        docs = vector_store.similarity_search(q, k=k)
        for d in docs:
            key = d.metadata.get("article", "") + d.metadata.get("law", "")
            if key in seen:
                continue
            seen.add(key)
            results.append({
                "law": d.metadata.get("law"),
                "article": d.metadata.get("article"),
                "title": d.metadata.get("title"),
                "content": d.page_content,
            })

    return {"search_results": results}

for q, d in [("명예훼손 처벌은?", "easy"),
             ("이혼과 손해배상 청구가 함께 가능해?", "hard")]:
    state = {"query": q, "difficulty": d, "messages": [], "search_results": [],
             "relevance_scores": [], "grade": "", "refinement_count": 0, "answer": ""}
    out = adaptive_search_node(state)
    print(f"  [{d}] '{q}' → {len(out['search_results'])}개")

---
## 문제 4: Self RAG — 관련성 평가 노드

검색된 각 문서가 질문에 얼마나 관련있는지 LLM이 점수화합니다.

**요구사항:**
- 함수 시그니처: `relevance_node(state) -> dict`
- 각 `search_results`에 대해:
  - LLM에게 `0.0~1.0` 점수 요청 (질문과 관련도)
  - 프롬프트: "다음 문서가 질문에 답하는 데 얼마나 유용한가요? 0.0(무관)~1.0(매우 관련) 숫자만 답하세요."
- 반환: `{"relevance_scores": [float, float, ...]}`
- LLM 응답 파싱 실패 시 기본값 `0.5`

**평가기준:**
- 결과 길이 == search_results 길이
- 모든 점수가 0.0 ~ 1.0 범위

In [ ]:
# 문제 4: relevance_node
def relevance_node(state):
    """각 검색 결과의 관련성을 0~1로 평가."""
    query = state["query"]
    results = state["search_results"]
    scores = []

    for r in results:
        # ---- 여기에 코드 작성 ----
        # 1) LLM에게 관련성 점수 요청
        # 2) 응답을 float로 파싱 (실패 시 0.5)
        # 3) scores에 append
        pass

    return {"relevance_scores": scores}

# 테스트
state = {"query": "명예훼손 처벌", "difficulty": "easy", "messages": [],
         "search_results": [
             {"law": "형법", "article": "제307조", "title": "명예훼손", "content": "..."},
             {"law": "형법", "article": "제329조", "title": "절도", "content": "..."},
         ],
         "relevance_scores": [], "grade": "", "refinement_count": 0, "answer": ""}
out = relevance_node(state)
print(f"관련성 점수: {out['relevance_scores']}")

In [ ]:
# ✅ 문제 4 정답
class _RelevanceScore(BaseModel):
    score: float = Field(ge=0.0, le=1.0, description="0.0(무관)~1.0(매우 관련)")

_relevance_scorer = llm.with_structured_output(_RelevanceScore)

def relevance_node(state):
    query = state["query"]
    results = state["search_results"]
    scores = []
    for r in results:
        prompt = f"""다음 문서가 질문에 답하는 데 얼마나 유용한가요? 0.0(무관)~1.0(매우 관련)으로 평가하세요.

질문: {query}
문서: {r.get('content', '')[:400]}"""
        try:
            s = _relevance_scorer.invoke([HumanMessage(content=prompt)]).score
        except Exception:
            s = 0.5
        scores.append(s)
    return {"relevance_scores": scores}

state = {"query": "명예훼손 처벌", "difficulty": "easy", "messages": [],
         "search_results": [
             {"law": "형법", "article": "제307조", "title": "명예훼손",
              "content": "공연히 사실을 적시하여 사람의 명예를 훼손한 자는 2년 이하의 징역이나 500만원 이하의 벌금에 처한다."},
             {"law": "형법", "article": "제329조", "title": "절도",
              "content": "타인의 재물을 절취한 자는 6년 이하의 징역 또는 1천만원 이하의 벌금에 처한다."},
         ],
         "relevance_scores": [], "grade": "", "refinement_count": 0, "answer": ""}
print("관련성 점수:", relevance_node(state)["relevance_scores"])

---
## 문제 5: Self RAG — 답변 품질 평가 + 재생성 루프

생성된 답변이 충분한 품질인지 LLM이 자가 평가합니다. 미달 시 다시 검색·생성.

**요구사항:**
- `generate_node(state)`: 관련성 상위 문서만 사용해 답변 생성 (threshold=0.5 이상)
- `grade_answer(state)`: 답변이 충분한지 `"pass"` 또는 `"fail"` 판정
- 라우터 `should_regenerate(state)`:
  - grade=="fail" AND refinement_count < 2 → 재검색 (`"retry"`)
  - 그 외 → 종료 (`END`)
- `refine_node(state)`: refinement_count += 1 반환

**평가기준:**
- generate_node 결과에 `answer` 필드 채워짐
- grade가 pass/fail 중 하나
- 재시도 로직으로 최대 2번까지 재생성

In [ ]:
# 문제 5: 답변 생성 + 품질 평가 + 재생성 루프
def generate_node(state):
    """관련성 높은 문서로 답변 생성."""
    query = state["query"]
    results = state["search_results"]
    scores = state.get("relevance_scores", [])
    # ---- 여기에 코드 작성 ----
    # 1) score >= 0.5인 문서만 context로 사용
    # 2) LLM에 답변 요청
    # 3) {"answer": ..., "messages": [AIMessage(...)]} 반환
    return {"answer": "미구현", "messages": [AIMessage(content="미구현")]}

def grade_answer(state):
    """답변 품질을 pass/fail로 평가."""
    query = state["query"]
    answer = state["answer"]
    # ---- 여기에 코드 작성 ----
    # LLM에게 "답변이 질문에 충분히 답하는가? pass/fail 중 하나만" 요청
    return {"grade": "pass"}

def refine_node(state):
    """재시도 카운터 증가."""
    return {"refinement_count": state.get("refinement_count", 0) + 1}

def should_regenerate(state):
    """재생성 여부 결정."""
    # ---- 여기에 코드 작성 ----
    return END

# 테스트 (단독 노드 호출)
state = {"query": "명예훼손 처벌", "difficulty": "easy", "messages": [],
         "search_results": [{"law": "형법", "article": "제307조", "title": "명예훼손",
                            "content": "형법 제307조 (명예훼손): 공연히 사실을 적시하여 사람의 명예를 훼손한 자는 2년 이하의 징역이나 500만원 이하의 벌금에 처한다."}],
         "relevance_scores": [0.9], "grade": "", "refinement_count": 0, "answer": ""}
out = generate_node(state)
print(f"답변: {out['answer'][:150]}")
state["answer"] = out["answer"]
print(f"평가: {grade_answer(state)['grade']}")

In [ ]:
# ✅ 문제 5 정답
RELEVANCE_THRESHOLD = 0.5
MAX_REFINEMENTS = 2

class _AnswerGrade(BaseModel):
    grade: Literal["pass", "fail"] = Field(description="답변이 질문에 충분히 답하는지")

_answer_grader = llm.with_structured_output(_AnswerGrade)

def generate_node(state):
    query = state["query"]
    results = state["search_results"]
    scores = state.get("relevance_scores", [])

    relevant = [r for r, s in zip(results, scores) if s >= RELEVANCE_THRESHOLD]
    if not relevant:
        relevant = results[:2]

    context = "\n\n".join([
        f"- {r.get('law','')} {r.get('article','')}: {r.get('content','')[:300]}"
        for r in relevant
    ])
    prompt = f"""다음 법률 자료를 참고하여 질문에 답하세요. 정확하고 명확하게 3-5문장.

질문: {query}

참고 자료:
{context}

답변:"""
    answer = llm.invoke([HumanMessage(content=prompt)]).content
    return {"answer": answer, "messages": [AIMessage(content=answer)]}

def grade_answer(state):
    query = state["query"]
    answer = state["answer"]
    prompt = f"""다음 답변이 질문에 충분히 답하는지 평가하세요. pass 또는 fail로만.

질문: {query}
답변: {answer}"""
    r = _answer_grader.invoke([HumanMessage(content=prompt)])
    return {"grade": r.grade}

def refine_node(state):
    return {"refinement_count": state.get("refinement_count", 0) + 1}

def should_regenerate(state):
    if state.get("grade") == "fail" and state.get("refinement_count", 0) < MAX_REFINEMENTS:
        return "retry"
    return END

state = {"query": "명예훼손 처벌", "difficulty": "easy", "messages": [],
         "search_results": [{"law": "형법", "article": "제307조", "title": "명예훼손",
                            "content": "형법 제307조 (명예훼손): 공연히 사실을 적시하여 사람의 명예를 훼손한 자는 2년 이하의 징역이나 500만원 이하의 벌금에 처한다."}],
         "relevance_scores": [0.9], "grade": "", "refinement_count": 0, "answer": ""}
out = generate_node(state)
print(f"답변: {out['answer'][:150]}")
state["answer"] = out["answer"]
print(f"평가: {grade_answer(state)['grade']}")

---
## 문제 6: Corrective RAG — 문서 등급화

검색 결과를 `relevance_scores`로 등급화하고, 기준 미달 시 `grade="fail"` 표시.

**요구사항:**
- 함수 시그니처: `corrective_grade_node(state) -> dict`
- 로직:
  - `max(relevance_scores)` 계산
  - 최고 점수가 `0.7` 이상이면 `"pass"`
  - 그 외에는 `"fail"` (외부 검색이 필요함을 시그널링)
- 반환: `{"grade": "pass" | "fail"}`

**평가기준:**
- 모든 문서 관련도 낮으면 fail
- 최소 하나라도 높으면 pass

In [ ]:
# 문제 6: corrective_grade_node
def corrective_grade_node(state):
    """검색 결과의 최고 관련성으로 pass/fail 판정."""
    scores = state.get("relevance_scores", [])
    # ---- 여기에 코드 작성 ----
    grade = "fail"
    return {"grade": grade}

# 테스트
for scores in [[0.9, 0.6, 0.4], [0.3, 0.2, 0.1], []]:
    state = {"query": "x", "difficulty": "", "messages": [], "search_results": [],
             "relevance_scores": scores, "grade": "", "refinement_count": 0, "answer": ""}
    print(f"  점수 {scores} → {corrective_grade_node(state)['grade']}")

In [ ]:
# ✅ 문제 6 정답
CORRECTIVE_THRESHOLD = 0.7

def corrective_grade_node(state):
    scores = state.get("relevance_scores", [])
    if not scores:
        grade = "fail"
    elif max(scores) >= CORRECTIVE_THRESHOLD:
        grade = "pass"
    else:
        grade = "fail"
    return {"grade": grade}

for scores in [[0.9, 0.6, 0.4], [0.3, 0.2, 0.1], []]:
    state = {"query": "x", "difficulty": "", "messages": [], "search_results": [],
             "relevance_scores": scores, "grade": "", "refinement_count": 0, "answer": ""}
    print(f"  점수 {scores} → {corrective_grade_node(state)['grade']}")

---
## 문제 7: Corrective RAG — 외부 폴백 검색

로컬 FAISS 검색 결과가 부족할 때 Wikipedia로 보완 검색을 수행합니다.

**요구사항:**
- Wikipedia 도구 생성 (`WikipediaAPIWrapper(lang="ko", top_k_results=2)`)
- 함수 시그니처: `external_search_node(state) -> dict`
- 기존 `search_results`에 Wikipedia 결과를 추가 (dict 리스트 동일 포맷, `law=None`, `title=wiki 제목`, `content=요약`)
- 라우터 `corrective_router(state)`:
  - grade=="fail" → `"external"`
  - grade=="pass" → `"generate"`

**평가기준:**
- external_search_node가 search_results를 확장
- Wikipedia 호출 실패 시 예외 없이 빈 리스트 유지

In [ ]:
# 문제 7: external_search_node (Wikipedia 폴백)
wiki_api = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=500, lang="ko")
wiki_tool = WikipediaQueryRun(api_wrapper=wiki_api)

def external_search_node(state):
    """Wikipedia로 외부 검색 수행 후 search_results에 추가."""
    query = state["query"]
    existing = state.get("search_results", [])
    # ---- 여기에 코드 작성 ----
    # 1) wiki_tool.invoke(query) 호출 (try/except 필요)
    # 2) 결과를 dict로 감싸 existing에 append
    # 3) {"search_results": existing}
    return {"search_results": existing}

def corrective_router(state):
    """grade에 따라 external / generate 분기."""
    # ---- 여기에 코드 작성 ----
    return "generate"

# 테스트
state = {"query": "탄핵 소추", "difficulty": "medium", "messages": [],
         "search_results": [], "relevance_scores": [], "grade": "fail",
         "refinement_count": 0, "answer": ""}
out = external_search_node(state)
print(f"외부 검색 후 문서 수: {len(out['search_results'])}")
for r in out["search_results"][:2]:
    print(f"  - {r.get('title', 'N/A')}: {str(r.get('content',''))[:80]}...")

In [ ]:
# ✅ 문제 7 정답
wiki_api = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=500, lang="ko")
wiki_tool = WikipediaQueryRun(api_wrapper=wiki_api)

def external_search_node(state):
    query = state["query"]
    existing = list(state.get("search_results", []))
    try:
        wiki_result = wiki_tool.invoke(query)
        if wiki_result and wiki_result.strip():
            existing.append({
                "law": None,
                "article": None,
                "title": f"Wikipedia: {query}",
                "content": wiki_result[:500],
            })
    except Exception as e:
        print(f"  ⚠️ Wikipedia 검색 실패: {e}")
    return {"search_results": existing}

def corrective_router(state):
    return "external" if state.get("grade") == "fail" else "generate"

state = {"query": "탄핵 소추", "difficulty": "medium", "messages": [],
         "search_results": [], "relevance_scores": [], "grade": "fail",
         "refinement_count": 0, "answer": ""}
out = external_search_node(state)
print(f"외부 검색 후 {len(out['search_results'])}개")
for r in out["search_results"][:2]:
    print(f"  - {r.get('title','N/A')}")

---
## 문제 8: 3종 RAG 통합 그래프

Adaptive + Self + Corrective 를 하나의 StateGraph로 통합하세요.

**요구사항:**
- 순서:
  1. `difficulty` (문제 2)
  2. `adaptive_search` (문제 3)
  3. `relevance` (문제 4)
  4. `corrective_grade` (문제 6)
  5. grade가 fail이면 → `external_search` → `relevance`로 돌아옴
  6. grade가 pass이면 → `generate` (문제 5)
  7. `grade_answer` → fail이면 `refine` → `adaptive_search`로 재시도 (최대 2회)
- `MemorySaver` 체크포인터 포함
- 함수 시그니처: `build_rag_graph() -> CompiledStateGraph`

**평가기준:**
- 다양한 난이도/주제 질문에 대해 결과 반환
- 외부 검색 폴백이 동작
- 재생성 루프가 무한 반복되지 않음 (refinement_count로 제한)

In [ ]:
# 문제 8: 3종 RAG 통합 그래프
def build_rag_graph():
    builder = StateGraph(RAGAgentState)
    # ---- 여기에 코드 작성 ----
    # 1) 노드 추가: difficulty, adaptive_search, relevance, corrective_grade,
    #              external_search, generate, grade_answer_node, refine
    # 2) START → difficulty → adaptive_search → relevance → corrective_grade
    # 3) corrective_grade에서 corrective_router로 분기: fail→external_search→relevance, pass→generate
    # 4) generate → grade_answer_node
    # 5) grade_answer_node → should_regenerate 분기: retry→refine→adaptive_search, end→END
    return None

# 통합 테스트
graph = build_rag_graph()
if graph:
    config = {"configurable": {"thread_id": "rag-test"}}
    test_queries = [
        "명예훼손 처벌은?",  # easy, 로컬에서 해결
        "배우자 부정행위 후 유기 시 이혼과 손해배상 함께 청구 가능?",  # hard
    ]
    for q in test_queries:
        print(f"\n👤 {q}")
        result = graph.invoke({
            "messages": [HumanMessage(content=q)], "query": q,
            "difficulty": "", "search_results": [], "relevance_scores": [],
            "grade": "", "refinement_count": 0, "answer": "",
        }, config=config | {"configurable": {"thread_id": f"rag-{q[:5]}"}})
        print(f"  난이도: {result['difficulty']}")
        print(f"  검색 문서: {len(result['search_results'])}, 재시도: {result['refinement_count']}")
        print(f"  🤖 {result['answer'][:200]}...")

In [ ]:
# ✅ 문제 8 정답
def build_rag_graph():
    builder = StateGraph(RAGAgentState)

    builder.add_node("difficulty", difficulty_node)
    builder.add_node("adaptive_search", adaptive_search_node)
    builder.add_node("relevance", relevance_node)
    builder.add_node("corrective_grade", corrective_grade_node)
    builder.add_node("external_search", external_search_node)
    builder.add_node("generate", generate_node)
    builder.add_node("grade_answer", grade_answer)
    builder.add_node("refine", refine_node)

    builder.add_edge(START, "difficulty")
    builder.add_edge("difficulty", "adaptive_search")
    builder.add_edge("adaptive_search", "relevance")
    builder.add_edge("relevance", "corrective_grade")

    # corrective_grade에서 분기
    builder.add_conditional_edges(
        "corrective_grade", corrective_router,
        {"external": "external_search", "generate": "generate"}
    )
    # 외부 검색 후 곧장 generate로 (무한루프 방지)
    builder.add_edge("external_search", "generate")

    # generate 후 품질 평가
    builder.add_edge("generate", "grade_answer")
    builder.add_conditional_edges(
        "grade_answer", should_regenerate,
        {"retry": "refine", END: END}
    )
    builder.add_edge("refine", "adaptive_search")

    return builder.compile(checkpointer=MemorySaver())

graph = build_rag_graph()
for q in ["명예훼손 처벌은?", "배우자 부정행위 후 유기 시 이혼·손해배상 동시 청구?"]:
    print(f"\n👤 {q}")
    result = graph.invoke({
        "messages": [HumanMessage(content=q)], "query": q,
        "difficulty": "", "search_results": [], "relevance_scores": [],
        "grade": "", "refinement_count": 0, "answer": "",
    }, config={"configurable": {"thread_id": q[:8]}})
    print(f"  난이도: {result['difficulty']}, 문서: {len(result['search_results'])}, 재시도: {result['refinement_count']}")
    print(f"  🤖 {result['answer'][:200]}...")


---
## 문제 9: `LegalRAGAgent` 통합 클래스

사용 편의를 위한 최종 클래스 API를 만듭니다.

**요구사항:**
- 초기화 시 `build_rag_graph()` 호출
- 메서드:
  - `ask(query, thread_id="default") -> dict`: 답변 + 난이도 + 사용 경로 + 재시도 횟수 반환
  - `inspect(thread_id) -> dict`: 해당 세션의 상태 스냅샷 (현재 단계, 점수, grade 등)
  - `visualize() -> str`: Mermaid 다이어그램
- 답변에 간단한 면책 조항 포함

**평가기준:**
- `ask()` 반환값에 `answer`, `difficulty`, `refinement_count`, `elapsed` 키
- `inspect()` 결과에 Weekend 3 state 필드 포함

In [ ]:
# 문제 9: LegalRAGAgent
class LegalRAGAgent:
    """Adaptive + Self + Corrective RAG 통합 에이전트."""

    DISCLAIMER = "\n\n⚠️ 본 답변은 일반 정보이며 법률 자문이 아닙니다."

    def __init__(self):
        # ---- 여기에 코드 작성 ----
        # self.graph = build_rag_graph()
        pass

    def ask(self, query, thread_id="default"):
        start = time.time()
        config = {"configurable": {"thread_id": thread_id}}
        # ---- 여기에 코드 작성 ----
        return {
            "answer": "미구현", "difficulty": "", "refinement_count": 0,
            "elapsed": 0, "thread_id": thread_id,
        }

    def inspect(self, thread_id="default"):
        """해당 세션 상태 스냅샷."""
        # ---- 여기에 코드 작성 ----
        return {}

    def visualize(self):
        # ---- 여기에 코드 작성 ----
        return ""

# 테스트
agent = LegalRAGAgent()
for q in ["절도죄 형량은?", "배우자가 부정행위 후 유기한 경우의 이혼과 손해배상"]:
    result = agent.ask(q, thread_id=q[:8])
    print(f"\n👤 {q}")
    print(f"  난이도: {result['difficulty']} | 재시도: {result['refinement_count']} | {result['elapsed']}초")
    print(f"  🤖 {result['answer'][:200]}...")

In [ ]:
# ✅ 문제 9 정답
class LegalRAGAgent:
    DISCLAIMER = "\n\n⚠️ 본 답변은 일반 정보이며 법률 자문이 아닙니다."

    def __init__(self):
        self.graph = build_rag_graph()

    def ask(self, query, thread_id="default"):
        start = time.time()
        config = {"configurable": {"thread_id": thread_id}}
        result = self.graph.invoke({
            "messages": [HumanMessage(content=query)], "query": query,
            "difficulty": "", "search_results": [], "relevance_scores": [],
            "grade": "", "refinement_count": 0, "answer": "",
        }, config=config)
        elapsed = round(time.time() - start, 2)
        answer = result["answer"] + self.DISCLAIMER
        return {
            "answer": answer, "difficulty": result["difficulty"],
            "refinement_count": result["refinement_count"],
            "elapsed": elapsed, "thread_id": thread_id,
        }

    def inspect(self, thread_id="default"):
        config = {"configurable": {"thread_id": thread_id}}
        state = self.graph.get_state(config)
        v = state.values
        return {
            "difficulty": v.get("difficulty"),
            "grade": v.get("grade"),
            "relevance_scores": v.get("relevance_scores"),
            "refinement_count": v.get("refinement_count"),
            "num_results": len(v.get("search_results", [])),
        }

    def visualize(self):
        try:
            return self.graph.get_graph().draw_mermaid()
        except Exception as e:
            return f"시각화 실패: {e}"

agent = LegalRAGAgent()
for q in ["절도죄 형량은?", "배우자 부정행위 후 유기 시 이혼·손해배상"]:
    result = agent.ask(q, thread_id=q[:8])
    print(f"\n👤 {q}")
    print(f"  난이도: {result['difficulty']} | 재시도: {result['refinement_count']} | {result['elapsed']}초")
    print(f"  🤖 {result['answer'][:200]}...")

---
## 문제 10: 미니 프로젝트 — 성능 벤치마크

10개 샘플 질문으로 `LegalRAGAgent`를 벤치마크하고 성능 리포트를 생성하세요.

**요구사항:**
- 테스트 쿼리 10개 (easy 3개, medium 4개, hard 3개)
- 각 쿼리 실행 → 난이도 분류 정확도, 평균 응답시간, 재시도 비율 집계
- `benchmark_report(agent, queries_with_expected) -> dict`:
  - `accuracy`: 난이도 분류 정확도 (%)
  - `avg_time`: 평균 응답시간 (초)
  - `avg_refinements`: 평균 재시도 횟수
  - `per_difficulty`: 난이도별 평균 시간
  - `queries`: 각 쿼리 상세

**평가기준:**
- 10개 쿼리 모두 실행 완료
- 리포트에 4개 핵심 지표 존재
- 난이도별 차이가 명확히 드러남



In [ ]:
# 문제 10: 성능 벤치마크
def benchmark_report(agent, queries_with_expected):
    """에이전트의 성능을 측정하고 리포트 생성."""
    # ---- 여기에 코드 작성 ----
    # 1) 각 쿼리 실행 → 난이도/시간/재시도 수집
    # 2) 정답 난이도와 비교하여 accuracy 계산
    # 3) 집계 후 dict 반환
    return {"accuracy": 0, "avg_time": 0, "avg_refinements": 0,
            "per_difficulty": {}, "queries": []}

# 테스트 쿼리 (난이도 라벨 포함)
benchmark_queries = [
    # easy (3)
    ("절도죄 형량은?", "easy"),
    ("명예훼손 처벌 뭐야?", "easy"),
    ("이혼 사유가 뭐가 있어?", "easy"),
    # medium (4)
    ("절도와 사기 차이점은?", "medium"),
    ("이사 선임은 어떻게 하고 임기는?", "medium"),
    ("불법행위와 채무불이행의 손해배상 차이?", "medium"),
    ("명예훼손과 허위사실 명예훼손의 구별?", "medium"),
    # hard (3)
    ("배우자 부정행위 후 유기 시 이혼·위자료·재산분할 절차는?", "hard"),
    ("사기죄로 재산상 이익 얻은 후 명예훼손까지 한 경우 가중 처벌?", "hard"),
    ("회사 이사가 고의로 손해 끼친 경우 이사 해임과 손해배상 동시에 가능?", "hard"),
]

agent = LegalRAGAgent()
report = benchmark_report(agent, benchmark_queries)

print(f"\n{'='*50}")
print(f"📊 LegalRAGAgent 성능 벤치마크")
print(f"{'='*50}")
print(f"정확도: {report.get('accuracy', 0):.1f}%")
print(f"평균 응답: {report.get('avg_time', 0):.2f}초")
print(f"평균 재시도: {report.get('avg_refinements', 0):.2f}")
print(f"난이도별 평균 시간: {report.get('per_difficulty', {})}")
print(f"총 쿼리: {len(report.get('queries', []))}개")

In [ ]:
# ✅ 문제 10 정답
def benchmark_report(agent, queries_with_expected):
    queries_detail = []
    correct = 0
    total_time = 0.0
    total_refinements = 0
    per_difficulty_times = {"easy": [], "medium": [], "hard": []}

    for q, expected in queries_with_expected:
        result = agent.ask(q, thread_id=f"bench-{hash(q) % 10000}")
        predicted = result["difficulty"]
        total_time += result["elapsed"]
        total_refinements += result["refinement_count"]
        per_difficulty_times.setdefault(predicted, []).append(result["elapsed"])

        if predicted == expected:
            correct += 1
        queries_detail.append({
            "query": q, "expected": expected, "predicted": predicted,
            "elapsed": result["elapsed"], "refinements": result["refinement_count"],
            "answer_preview": result["answer"][:100],
        })

    n = len(queries_with_expected)
    per_difficulty_avg = {
        k: round(sum(v)/len(v), 2) if v else 0
        for k, v in per_difficulty_times.items()
    }

    return {
        "accuracy": round(correct / n * 100, 1) if n else 0,
        "avg_time": round(total_time / n, 2) if n else 0,
        "avg_refinements": round(total_refinements / n, 2) if n else 0,
        "per_difficulty": per_difficulty_avg,
        "queries": queries_detail,
    }

benchmark_queries = [
    ("절도죄 형량은?", "easy"),
    ("명예훼손 처벌 뭐야?", "easy"),
    ("이혼 사유가 뭐가 있어?", "easy"),
    ("절도와 사기 차이점은?", "medium"),
    ("이사 선임은 어떻게 하고 임기는?", "medium"),
    ("불법행위와 채무불이행의 손해배상 차이?", "medium"),
    ("명예훼손과 허위사실 명예훼손의 구별?", "medium"),
    ("배우자 부정행위 후 유기 시 이혼·위자료·재산분할 절차는?", "hard"),
    ("사기죄로 재산상 이익 얻은 후 명예훼손까지 한 경우 가중 처벌?", "hard"),
    ("회사 이사가 고의로 손해 끼친 경우 이사 해임과 손해배상 동시에 가능?", "hard"),
]

agent = LegalRAGAgent()
report = benchmark_report(agent, benchmark_queries)

print(f"\n{'='*50}")
print(f"📊 LegalRAGAgent 성능 벤치마크")
print(f"{'='*50}")
print(f"정확도: {report['accuracy']}%")
print(f"평균 응답: {report['avg_time']}초")
print(f"평균 재시도: {report['avg_refinements']}")
print(f"난이도별 평균 시간: {report['per_difficulty']}")
print(f"\n상세 (첫 3개):")
for q in report["queries"][:3]:
    print(f"  [{q['expected']} → {q['predicted']}] {q['query'][:30]}... ({q['elapsed']}초, 재시도 {q['refinements']})")

---
## 🎉 Weekend 3 완료!

### 최종 달성한 것들 (3주간)

**Weekend 1 — Tool Calling**
- 법률 데이터셋 구축 + FAISS 벡터스토어
- 5개 도구 (`search_law`, `interpret_law`, `compare_articles`, `search_cases`, `explain_term`)
- 에이전트 루프 + 대화형 챗봇 + 면책 조항

**Weekend 2 — LangGraph**
- `LegalAgentState` + StateGraph
- 조건부 라우팅 + ReAct 패턴
- MemorySaver + Human-in-the-Loop

**Weekend 3 — Adaptive/Self/Corrective RAG**
- Adaptive Routing (난이도별 검색 전략)
- Self RAG (관련성·품질 자가 평가 + 재생성)
- Corrective RAG (문서 등급화 + 외부 폴백)
- 3종 통합 + 성능 벤치마크

### 다음 단계 아이디어
- 실제 법률 데이터(판례 DB)로 교체
- Gradio UI로 웹 서비스화
- Streaming 응답 + 토큰 단위 출력
- LangSmith로 에이전트 추적·디버깅